# SVD (Singular Value Decomposition)

Es un algoritmo de filtrado colaborativo basado en factorización matricial que representa usuarios e ítems mediante factores latentes aprendidos a partir de sus valoraciones, permitiendo predecir las preferencias de los usuarios sobre ítems que aún no ha valorado.

In [13]:
import pandas as pd
import numpy as np

In [14]:
df = pd.read_csv('resumen_usuario.csv')

C:\Users\esper\AppData\Local\Temp\ipykernel_3632\1565762017.py:1: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('resumen_usuario.csv')


In [4]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

reader = Reader(rating_scale=(1,10))
surprise_df = df[["User-ID", "ISBN", "Book-Rating"]].copy()
surprise_df.columns = ["uid", "iid", "rating"]

data = Dataset.load_from_df(surprise_df, reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

algo = SVD()
algo.fit(trainset)
predictions = algo.test(testset)

In [15]:
# Transformación a dataframe
preds_df = pd.DataFrame([
    {
        'User-ID': pred.uid,
        'ISBN': pred.iid,
        'r_ui': pred.r_ui,
        'est': pred.est
    }
    for pred in predictions
])

In [37]:

# 
from metrics import evaluar_metricas_usuario

K = 10
metricas_usuarios = (
    preds_df
    .groupby('User-ID')
    .apply(evaluar_metricas_usuario, k=K)
    .reset_index()
)

grupos_usuario = (
    df[['User-ID', 'Grupo_Etario', 'Grupo_Historial', 'Grupo_Exigencia']]
    .drop_duplicates('User-ID')
)

df_evaluacion = pd.merge(
    metricas_usuarios,
    grupos_usuario,
    on='User-ID',
    how='inner',
    validate='one_to_one'
)

In [40]:
# A. Por Grupo Etario (Demográfico)
print(f"=== Métricas por Grupo Etario (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Etario')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

# B. Por Historial de Interacciones (Comportamiento)
print(f"\n=== Métricas por Historial de Interacciones (0: Corto, 1: Largo) (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Historial')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

# C. Por Grado de Exigencia (Comportamiento)
print(f"\n=== Métricas por Grado de Exigencia (0: Exigente, 1: Normal, 2: Generoso) (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Exigencia')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

=== Métricas por Grupo Etario (K=10) ===
                   MAE      CG@10     DCG@10   NDCG@10
Grupo_Etario                                          
Adultos       1.348099  19.362830  13.033915  0.982367
Jóvenes       1.347780  16.515669  11.912382  0.986770
Mayores       1.310445  13.956989  10.660941  0.988830

=== Métricas por Historial de Interacciones (0: Corto, 1: Largo) (K=10) ===
                      MAE      CG@10     DCG@10   NDCG@10
Grupo_Historial                                          
0                1.387243   7.569976   7.569976  1.000000
1                1.348449  20.910721  13.813402  0.978732

=== Métricas por Grado de Exigencia (0: Exigente, 1: Normal, 2: Generoso) (K=10) ===
                      MAE      CG@10     DCG@10   NDCG@10
Grupo_Exigencia                                          
0                2.543366   8.084031   6.497840  0.987983
1                1.134678  18.047733  12.515771  0.983329
2                1.841978  17.552894  13.400827  0.996636

Obserbaciones:

Grupo Etario:

Representa el agrupamiento más homogéneo. Si bien se observan ciertas diferencias entre los tres grupos, estos son relativamente pequeñas. Estas diferencias podrían estar relacionadas con la cantidad de valoraciones disponibles para cada grupo, ya que un número desigual de observaciones puede afectar la estabilidad de la métricas obtenidas.


Grupo separado por historial:

El sistema presenta una diferencia marcada en su desempeño según la longitud del historial de los usuarios, favoreciendo considerablemente aquellos con un historial largo. El valor de NDGC = 1 indica que, en los casos evaluados, los ítems relevantes fueron posicionados coreectamente. Sin embargo, este resultado debe tomarse con precaución, ya que muchos usuarios poseen muy pocos ítems en el conjunto de prueba, incluso en algunos casos solo uno, lo que puede hacer que alcanzar un NDCG alto sea sencillo.


Grado de exigencia:

El sistema presenta dificultades para estimar con precisión las calificaciones de los usuarios exigentes (Grupo 0), sobreestimando sus valoraciones y empeorando el error (MAE). En contraste, los usuarios normales y generosos (Grupo 1 y 2) resultan altamente beneficiados, obteniendo los mejores valores en todas las métricas debido a la menor varianza de sus evaluaciones. Destacando que los usuarios normales tienen mejor MAE.

### Test t para grupos separados por historial

In [41]:
from scipy import stats

# Separar los usuarios en dos submuestras según su historial
g_corto = df_evaluacion[df_evaluacion['Grupo_Historial'] == 0]
g_largo = df_evaluacion[df_evaluacion['Grupo_Historial'] == 1]

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

print("="*60)
print(" PRUEBA T DE STUDENT (WELCH) - HISTORIAL CORTO VS. LARGO")
print("="*60)

for metrica in metricas:
    # Extracción de valores limpios
    v_corto = g_corto[metrica].dropna()
    v_largo = g_largo[metrica].dropna()
    
    # Ejecución del T-test
    t_stat, p_val = stats.ttest_ind(v_corto, v_largo, equal_var=False)
    
    # Interpretación del p-value (umbral alpha = 0.05)
    es_significativo = "SÍ (Diferencia estadísticamente significativa)" if p_val < 0.05 else "NO (Sin evidencia de diferencia)"
    
    print(f"\n[Métrica: {metrica}]")
    print(f"  • Promedio (Historial Corto) : {v_corto.mean():.4f}")
    print(f"  • Promedio (Historial Largo) : {v_largo.mean():.4f}")
    print(f"  • Estadístico t              : {t_stat:.4f}")
    print(f"  • p-value                    : {p_val:.4e}")
    print(f"  • ¿Existe inequidad?         : {es_significativo}")


 PRUEBA T DE STUDENT (WELCH) - HISTORIAL CORTO VS. LARGO

[Métrica: MAE]
  • Promedio (Historial Corto) : 1.3872
  • Promedio (Historial Largo) : 1.3484
  • Estadístico t              : 2.7101
  • p-value                    : 6.7368e-03
  • ¿Existe inequidad?         : SÍ (Diferencia estadísticamente significativa)

[Métrica: CG@10]
  • Promedio (Historial Corto) : 7.5700
  • Promedio (Historial Largo) : 20.9107
  • Estadístico t              : -82.9170
  • p-value                    : 0.0000e+00
  • ¿Existe inequidad?         : SÍ (Diferencia estadísticamente significativa)

[Métrica: DCG@10]
  • Promedio (Historial Corto) : 7.5700
  • Promedio (Historial Largo) : 13.8134
  • Estadístico t              : -87.3613
  • p-value                    : 0.0000e+00
  • ¿Existe inequidad?         : SÍ (Diferencia estadísticamente significativa)

[Métrica: NDCG@10]
  • Promedio (Historial Corto) : 1.0000
  • Promedio (Historial Largo) : 0.9787
  • Estadístico t              : 66.2502
  • p-value

c:\Users\esper\Desktop\tp-recomendacion\venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:601: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


La prueba T confirma diferencias estadísticamente significativas en todas las métricas evaluadas entre usuarios de historial corto y largo.

El modelo comete un error mayor al predecir notas para usuarios con pocas interacciones (posibles arranques en frío)

Caso de NDCG: Si bien la prueba T indica un valor mayor para el historial corto, esta métrica sufre de un sesgo por la escasez y la varianza en el grupo.

### ANOVA para grupos etarios

In [42]:
from scipy import stats

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

print("="*60)
print(" TEST ANOVA: GRUPOS ETARIOS")
print("="*60)

for metrica in metricas:
    # 1. Agrupar las muestras por cada rango etario
    grupos = [
        datos[metrica].dropna() 
        for _, datos in df_evaluacion.groupby('Grupo_Etario')
    ]
    
    # 2. ANOVA de un factor
    f_stat, p_val = stats.f_oneway(*grupos)
    
    es_significativo = p_val < 0.05
    conclusion = "SÍ (Diferencias significativas entre edades)" if es_significativo else "NO (Métrica homogénea entre edades)"
    
    print(f"\n[Métrica: {metrica}]")
    print(f"  • Estadístico F    : {f_stat:.4f}")
    print(f"  • p-value          : {p_val:.4e}")
    print(f"  • ¿Existe inequidad?: {conclusion}")

 TEST ANOVA: GRUPOS ETARIOS

[Métrica: MAE]
  • Estadístico F    : 0.3644
  • p-value          : 6.9463e-01
  • ¿Existe inequidad?: NO (Métrica homogénea entre edades)

[Métrica: CG@10]
  • Estadístico F    : 37.0724
  • p-value          : 8.6321e-17
  • ¿Existe inequidad?: SÍ (Diferencias significativas entre edades)

[Métrica: DCG@10]
  • Estadístico F    : 33.6956
  • p-value          : 2.4906e-15
  • ¿Existe inequidad?: SÍ (Diferencias significativas entre edades)

[Métrica: NDCG@10]
  • Estadístico F    : 21.2414
  • p-value          : 6.1228e-10
  • ¿Existe inequidad?: SÍ (Diferencias significativas entre edades)


El modelo comete el mismo nivel de error promedio al predecir las calificaciones independientemente del grupo. En cuanto al resto de las métricas, existen diferencias estadísticamente significativas. Las cuales pueden surgir de la diferencia en las cantidades de iteraciones de cada grupo.



### ANOVA para grupos separados por exigencia

In [43]:
from scipy import stats

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

print("="*60)
print(" TEST ANOVA - GRADO DE EXIGENCIA")
print("="*60)

for metrica in metricas:
    # 1. Agrupar muestras por grupo de exigencia (0: Exigente, 1: Normal, 2: Generoso)
    grupos = [
        datos[metrica].dropna() 
        for _, datos in df_evaluacion.groupby('Grupo_Exigencia')
    ]
    
    # 2. ANOVA de un factor
    f_stat, p_val = stats.f_oneway(*grupos)
    
    es_significativo = p_val < 0.05
    conclusion = "SÍ (Diferencias significativas según exigencia)" if es_significativo else "NO (Métrica homogénea)"
    
    print(f"\n[Métrica: {metrica}]")
    print(f"  • Estadístico F    : {f_stat:.4f}")
    print(f"  • p-value          : {p_val:.4e}")
    print(f"  • ¿Existe inequidad?: {conclusion}")

 TEST ANOVA - GRADO DE EXIGENCIA

[Métrica: MAE]
  • Estadístico F    : 3991.9798
  • p-value          : 0.0000e+00
  • ¿Existe inequidad?: SÍ (Diferencias significativas según exigencia)

[Métrica: CG@10]
  • Estadístico F    : 359.2116
  • p-value          : 1.3519e-154
  • ¿Existe inequidad?: SÍ (Diferencias significativas según exigencia)

[Métrica: DCG@10]
  • Estadístico F    : 753.8898
  • p-value          : 6.4326e-319
  • ¿Existe inequidad?: SÍ (Diferencias significativas según exigencia)

[Métrica: NDCG@10]
  • Estadístico F    : 153.0330
  • p-value          : 8.5134e-67
  • ¿Existe inequidad?: SÍ (Diferencias significativas según exigencia)


Se confirma que existe una diferencia estadisticamente significativa. El modelo castiga fuertmenete a aquellos usuarios que son exigentes a la hora de dar su evaluación.

# Conclusión general del SVD

El modelo SVD logra una buena capacidad de ordenación (NDCG cercano a 1). Esto significa que SVD puede ser una buena opción cuando el objetivo principal es el ranking de recomendaciones, no la estimación exacta del puntaje.